In [ ]:
MAPPING FILE: mplt_CDM_BATCH_ID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_BATCH_ID
Migrated from IICS mapping: mplt_CDM_BATCH_ID
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table" (e.g., "CDM.CDM_BATCH_CTRLID").
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMBatchIDMapplet:
    """
    ETL Pipeline for processing batch IDs.

    Sources: CDM.CDM_BATCH_CTRLID
    Targets: Processed Batch ID and Source Name
    Transformation Logic: Lookup maximum batch ID for a given source name, handle nulls, and output processed data.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source.

        IICS Source Qualifier equivalent.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('source_table', 'CDM.CDM_BATCH_CTRLID')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(
            table_name=table_name,
            local_file_path=local_file_path
        )

        log_df_info(df, "Source: CDM.CDM_BATCH_CTRLID")
        logger.info("Data extraction complete")
        return df

    def transform(self, df: DataFrame) -> DataFrame:
        """
        Apply business transformations.

        IICS Transformation Logic equivalent.
        """
        logger.info("Applying transformations")

        # Step 1: Mapplet Initialization (Pass-through SOURCE_NAME)
        df = df.select(
            F.trim(F.col("SOURCE_NAME")).alias("SOURCE_NAME"),
            F.col("BATCH_ID")
        )

        # Step 2: Lookup Batch ID
        lookup_table_name = self.config.get('lookup_table', 'CDM.CDM_BATCH_CTRLID')
        lookup_local_path = self.config.get('lookup_local_path', None)

        lookup_df = read_table(
            table_name=lookup_table_name,
            local_file_path=lookup_local_path
        ).select(
            F.trim(F.col("SOURCE_NAME")).alias("SOURCE_NAME"),
            F.col("BATCH_ID")
        )

        # Perform lookup using broadcast join
        df = df.join(
            F.broadcast(lookup_df),
            on="SOURCE_NAME",
            how="left"
        )

        # Step 3: Null Check and Batch ID Assignment
        df = df.withColumn(
            "o_BATCH_ID",
            F.when(F.col("BATCH_ID").isNull(), F.lit(-999)).otherwise(F.col("BATCH_ID"))
        ).select(
            "o_BATCH_ID",
            "SOURCE_NAME"
        )

        log_df_info(df, "After transformations")
        return df

    def load(self, df: DataFrame):
        """
        Load data to target.

        IICS Target equivalent.
        """
        logger.info("Starting data load")

        target_path = self.config.get('target_path', '/path/to/target')

        log_df_info(df, "Before load")

        (df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'overwrite'))
            .option("overwriteSchema", "true")
            .save(target_path)
        )

        # Optimize target table
        if self.config.get('optimize_target', True):
            logger.info("Optimizing target table")
            zorder_cols = self.config.get('zorder_columns', [])
            if zorder_cols:
                self.spark.sql(f"""
                    OPTIMIZE delta.`{target_path}`
                    ZORDER BY ({', '.join(zorder_cols)})
                """)

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            source_df = self.extract()

            # Transform
            transformed_df = self.transform(source_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'CDM.CDM_BATCH_CTRLID',
    'source_local_path': r'path\to\source.csv',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'overwrite',
    'zorder_columns': ['SOURCE_NAME'],

    # Lookup configuration
    'lookup_table': 'CDM.CDM_BATCH_CTRLID',
    'lookup_local_path': r'path\to\lookup.csv',

    # Performance configuration
    'shuffle_partitions': 200,
    'optimize_target': True
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("ETL_Pipeline_mplt_CDM_BATCH_ID").getOrCreate()
    pipeline = CDMBatchIDMapplet(spark, config)
    pipeline.execute()
MAPPING FILE: m_CDM_W_CLAIM_CD_SCD3_IU.txt
====================================================================================================

"""
ETL Pipeline: m_CDM_W_CLAIM_CD_SCD3_IU
Migrated from IICS mapping: m_CDM_W_CLAIM_CD_SCD3_IU
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ
    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMClaimSCD3Pipeline:
    """
    ETL Pipeline for processing SCD Type 3 logic for claim data.

    Sources: CDH_GW_BUR
    Targets: W_CLAIM_CD_BUR_SCD3_I (Insert), W_CLAIM_CD_BUR_SCD3_U (Update)
    Transformation Logic: SCD Type 3 handling for BUR field.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }
        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source table CDH_GW_BUR.
        """
        logger.info("Starting data extraction")
        table_name = self.config.get('source_table', 'catalog.database.table')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(table_name=table_name, local_file_path=local_file_path)
        log_df_info(df, "Source: CDH_GW_BUR")
        return df

    def transform(self, df: DataFrame) -> DataFrame:
        """
        Apply transformations including expressions, lookups, and flag evaluation.
        """
        logger.info("Applying transformations")

        # Step 2: Expression Transformation
        df = df.select(
            F.col("POLICY_STATE").alias("INTEGRATION_ID"),
            F.col("BUR"),
            F.lit("GWCDH").alias("SOURCE_NAME")
        )
        log_df_info(df, "After Expression Transformation")

        # Step 3: Lookup Transformation
        lookup_table_name = self.config.get('lookup_table', 'catalog.database.lookup_table')
        lookup_local_path = self.config.get('lookup_local_path', None)
        lookup_df = read_table(table_name=lookup_table_name, local_file_path=lookup_local_path)

        lookup_df = lookup_df.select("LKP_ROW_WID", "LKP_INTEGRATION_ID", "LKP_NEW_BUR")
        df = df.join(
            F.broadcast(lookup_df),
            df.INTEGRATION_ID == lookup_df.LKP_INTEGRATION_ID,
            "left"
        ).select(
            df["*"],
            lookup_df["LKP_ROW_WID"],
            lookup_df["LKP_NEW_BUR"]
        )
        log_df_info(df, "After Lookup Transformation")

        # Step 4: Flag Evaluation
        df = df.withColumn(
            "o_Flag",
            F.when(F.col("LKP_ROW_WID").isNull(), "I")
            .when(F.md5(F.col("BUR")) == F.md5(F.col("LKP_NEW_BUR")), "NC")
            .otherwise("U")
        ).withColumn(
            "CDM_INSERT_DT", F.current_timestamp()
        ).withColumn(
            "CDM_UPDATE_DT", F.current_timestamp()
        ).withColumn(
            "TGT_TABLE_NAME", F.lit("W_CLAIM_CD_BUR_SCD3")
        )
        log_df_info(df, "After Flag Evaluation")

        return df

    def load(self, df: DataFrame):
        """
        Load data into target tables for insert and update operations.
        """
        logger.info("Starting data load")

        # Step 5: Router Transformation
        insert_df = df.filter(F.col("o_Flag") == "I").select("o_Flag", "CDM_INSERT_DT", "TGT_TABLE_NAME", "BUR")
        update_df = df.filter(F.col("o_Flag") == "U").select("o_Flag", "CDM_UPDATE_DT", "TGT_TABLE_NAME", "BUR")

        # Step 7: Target Load (Insert)
        insert_target_path = self.config.get('insert_target_path', '/path/to/insert_target')
        (insert_df.write
            .format("delta")
            .mode("append")
            .save(insert_target_path)
        )
        logger.info(f"Inserted records written to {insert_target_path}")

        # Step 8: Target Load (Update)
        update_target_path = self.config.get('update_target_path', '/path/to/update_target')
        (update_df.write
            .format("delta")
            .mode("append")
            .save(update_target_path)
        )
        logger.info(f"Updated records written to {update_target_path}")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")
        try:
            # Extract
            source_df = self.extract()

            # Transform
            transformed_df = self.transform(source_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")
        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise

# Configuration
config = {
    'source_table': 'CDH_GW_BUR',
    'source_local_path': r'/path/to/source.csv',
    'lookup_table': 'CDM.W_CLAIM_CD_BUR_SCD3',
    'lookup_local_path': r'/path/to/lookup.csv',
    'insert_target_path': '/path/to/insert_target',
    'update_target_path': '/path/to/update_target',
    'shuffle_partitions': 200
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDMClaimSCD3Pipeline").getOrCreate()
    pipeline = CDMClaimSCD3Pipeline(spark, config)
    pipeline.execute()
MAPPING FILE: mplt_CDM_ROW_WID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_ROW_WID
Migrated from IICS mapping: mplt_CDM_ROW_WID

Purpose:
This pipeline retrieves the maximum ROW_WID and associated TABLE_NAME from a target table using a lookup transformation,
calculates a new ROW_WID based on conditional logic, and outputs the result to a Snowflake table.

"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMRowWidPipeline:
    """
    ETL Pipeline for mplt_CDM_ROW_WID.

    Sources: Custom Table
    Targets: Snowflake Table
    Transformation Logic: Lookup maximum ROW_WID, calculate new ROW_WID, and output to target.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source.

        IICS Source Qualifier equivalent.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('lookup_table', 'catalog.database.table')
        local_file_path = self.config.get('lookup_local_path', None)

        df = read_table(
            table_name=table_name,
            local_file_path=local_file_path
        )

        log_df_info(df, "Source: Lookup Table")
        logger.info("Data extraction complete")
        return df

    def transform(self, lookup_df: DataFrame) -> DataFrame:
        """
        Apply business transformations.

        IICS Transformation Logic equivalent.
        """
        logger.info("Applying transformations")

        # Lookup logic: Retrieve maximum ROW_WID and TABLE_NAME
        max_row_wid_df = lookup_df.groupBy("TABLE_NAME").agg(
            F.coalesce(F.max("ROW_WID"), F.lit(0)).alias("ROW_WID")
        )

        log_df_info(max_row_wid_df, "After Lookup Transformation")

        # Calculate new ROW_WID
        new_row_wid_df = max_row_wid_df.withColumn(
            "ROW_WID",
            F.col("ROW_WID") + F.lit(1)
        )

        log_df_info(new_row_wid_df, "After ROW_WID Calculation")
        return new_row_wid_df

    def load(self, df: DataFrame):
        """
        Load data to target.

        IICS Target equivalent.
        """
        logger.info("Starting data load")

        target_path = self.config.get('target_path', 'path/to/target')

        log_df_info(df, "Before load")

        (df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'overwrite'))
            .option("overwriteSchema", "true")
            .save(target_path)
        )

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            lookup_df = self.extract()

            # Transform
            transformed_df = self.transform(lookup_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Lookup configuration
    'lookup_table': 'catalog.database.lookup_table',
    'lookup_local_path': r'path/to/lookup.csv',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'overwrite',

    # Performance configuration
    'shuffle_partitions': 200
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDM_ROW_WID_Pipeline").getOrCreate()
    pipeline = CDMRowWidPipeline(spark, config)
    pipeline.execute()